# Cheat Sheet 7: Imputing Missing Data

_MMA 860 – Management of Data_

## What this covers

- Patterns of missingness: MCAR, MAR, MNAR
- Multiple Imputation by Chained Equations (MICE) with statsmodels
- Pooling regression results across imputed datasets
- A brief note on Little's test for MCAR

## Working with AI

When you spot missing values, ask your AI assistant: *"Given this missingness pattern, would you guess the data is MCAR, MAR, or MNAR? What's your reasoning?"* It will not always be right, but the reasoning helps you think it through.

AI is also handy for boilerplate – pooling logic, plotting missingness patterns – just verify the math against the textbook.

## A quick refresher: when to impute

Imputation is only valid when missingness is **MCAR** (missing completely at random) or **MAR** (missing at random conditional on observed variables). If data is **MNAR** (missing not at random), no imputation method will give you unbiased estimates, and you should treat the missingness itself as informative.

In practice, you rarely know for certain. Look at the pattern of missingness, talk to whoever generated the data, and document your assumption in your report.

# Importing Our Data and Running a Regression

In [1]:
#First we import our data from the appropriate file
import pandas as pd
import os.path as osp
import numpy as np
data_path = osp.join(osp.curdir,'Data','Cheat_Sheet_Missing_Data_V2_0.xlsx')
data = pd.read_excel(data_path,sheet_name=0)
# Drop Obs column
data = data.drop(columns = 'Obs')
#Note that Family_Income has only 977 values where all the rest have 1000
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Grocery_Bill       1000 non-null   float64
 1   N_Adults           1000 non-null   int64  
 2   Family_Income      977 non-null    float64
 3   Family_Size        1000 non-null   int64  
 4   N_Vehicles         1000 non-null   int64  
 5   Distance_to_Store  1000 non-null   int64  
 6   Vegetarian         1000 non-null   int64  
 7   N_Children         1000 non-null   int64  
 8   Family_Pet         1000 non-null   int64  
dtypes: float64(2), int64(7)
memory usage: 70.4 KB


In [2]:
# View the null Family_Income rows
data[(pd.isnull(data.Family_Income))]

,Grocery_Bill,N_Adults,Family_Income,Family_Size,N_Vehicles,Distance_to_Store,Vegetarian,N_Children,Family_Pet
11,153.203795,1,NaN,1,2,1,0,0,0
27,121.657482,1,NaN,1,0,3,0,0,0
55,167.129757,1,NaN,1,1,6,0,0,1
72,336.694767,3,NaN,3,3,2,0,0,1
93,272.120366,2,NaN,2,1,1,0,0,0
123,391.558272,2,NaN,5,2,4,0,3,0
143,165.372652,1,NaN,1,1,2,0,0,0
161,222.134584,1,NaN,1,3,5,0,0,0
182,113.109780,1,NaN,1,3,1,0,0,0
201,231.710796,1,NaN,1,1,16,0,0,0


In [3]:
# We will run a regression with all fields except Family_Size
frmla = 'Grocery_Bill ~ N_Adults + Family_Income + N_Vehicles + ' + \
    'Distance_to_Store + Vegetarian + N_Children + Family_Pet'

In [4]:
#Train and view our model
from statsmodels.formula.api import ols
results = ols(frmla,data).fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           Grocery_Bill   R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     723.7
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        19:22:38   Log-Likelihood:                -4788.7
No. Observations:                 977   AIC:                             9593.
Df Residuals:                     969   BIC:                             9633.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            23.0180      5.798      3.970      0.000      11.640      34.396
N_Adults             54.5158      3.878     14.057      0.000      46.905      62.126
Family_Income         0.0010   8.89e-05     10.718      0.000       0.001       0.001
N_Vehicles           -1.0368      1.167     -0.888      0.375      -3.327       1.254
Distance_to_Store     3.8825      0.193     20.151      0.000       3.504       4.261
Vegetarian           -7.9646      4.264     -1.868      0.062     -16.331       0.402
N_Children           28.1890      1.231     22.899      0.000      25.773      30.605
Family_Pet            1.3033      2.875      0.453      0.650      -4.339       6.946
==============================================================================
Omnibus:                       51.282   Durbin-Watson:                   2.089
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              172.930
Skew:                           0.084   Prob(JB):                     2.81e-38
Kurtosis:                       5.054   Cond. No.                     6.23e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.23e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

# Performing MICE

*Depending on your version of Anaconda, you may get a notification about a function in MICE being deprecated. This means that that function may not work in a future Python update.*

The following snippet of code performs the imputation on the dataset.

In [5]:
#Import the required libraries
import statsmodels.imputation.mice as mice
import statsmodels.regression.linear_model as sm

#Impute the data set with MICE Data
imp = mice.MICEData(data)
imputed_data = imp.next_sample()

#Merge the imputed data with the blank data
merge_data = pd.merge(imputed_data,data,left_index=True,right_index=True)
merge_data = merge_data[['Family_Income_x','Family_Income_y']]
#Family_Income_x are the imputed values
merge_data[(pd.isnull(merge_data.Family_Income_y))]

,Family_Income_x,Family_Income_y
11,89203.0,NaN
27,75906.0,NaN
55,72391.0,NaN
72,120719.0,NaN
93,130138.0,NaN
123,145583.0,NaN
143,90217.0,NaN
161,89335.0,NaN
182,82485.0,NaN
201,78245.0,NaN


Selecting a single imputation is not the best way to use imputed data for predictive modelling. Instead, you want to use all the imputed data sets and pool the results. To do this, you need to run a regression on all 5 imputed data sets, and then pool the regression results. This can be done by setting the ${n\_imputations}$ argument.

In [6]:
mice = mice.MICE(frmla, sm.OLS, imp)
results = mice.fit(n_imputations=5)
print(results.summary())

                              Results: MICE
Method:                   MICE              Sample size:          1000   
Model:                    OLS               Scale                 1067.76
Dependent variable:       Grocery_Bill      Num. imputations      5      
-------------------------------------------------------------------------
                   Coef.  Std.Err.    t    P>|t|   [0.025   0.975]  FMI  
-------------------------------------------------------------------------
Intercept         23.7055   5.7663  4.1110 0.0000  12.4038 35.0073 0.0116
N_Adults          54.7700   3.8663 14.1660 0.0000  47.1922 62.3479 0.0159
Family_Income      0.0009   0.0001 10.4850 0.0000   0.0008  0.0011 0.0242
N_Vehicles        -0.7955   1.1529 -0.6901 0.4902  -3.0551  1.4640 0.0040
Distance_to_Store  3.8899   0.1911 20.3537 0.0000   3.5153  4.2645 0.0008
Vegetarian        -7.7947   4.2604 -1.8296 0.0673 -16.1449  0.5554 0.0002
N_Children        28.3937   1.2202 23.2703 0.0000  26.0022 30.7852 0

## Interpreting Results

You can interpret these regression results as you normally would. 
If you are interested in learning more about MICE and multiple imputation, this is an excellent paper:
https://www.jstatsoft.org/article/view/v045i03


## Summary

Here is how an imputation works, step by step:

  1) We set the number of imputations we want. If we set m = 5, mice creates 5 exact copies of the original dataset
  
  2) Then, in the background, a regression model is built to predict the missing data (in this case, Family Income) using all the other variables available
  
  3) For each of the five datasets, we use the model just created to predict the missing values. In each of the datasets, the error term of the regression is randomized. This means that each of the 5 data sets will have different imputed values!
  
  4) We then run statistical tests on all of the 5 data sets, and pool the results together for a more robust answer


## Little's Test

There is a formal test for determining whether data is MAR or MCAR – Little’s test. You can use the test by installing the impyute package.

## Try it yourself

1. Drop rows with missing income and rerun the regression. Compare the coefficients to the imputed version.
2. Increase `n_imputations` to 10. Do the standard errors change?
3. **AI exercise.** Ask your AI tool: *"Walk me through what mean imputation does to the variance of a coefficient estimate."* Compare its answer to the textbook.

In [12]:
data_drop = pd.read_excel(data_path,sheet_name=0)

In [13]:
data_drop.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Obs                1000 non-null   int64  
 1   Grocery_Bill       1000 non-null   float64
 2   N_Adults           1000 non-null   int64  
 3   Family_Income      977 non-null    float64
 4   Family_Size        1000 non-null   int64  
 5   N_Vehicles         1000 non-null   int64  
 6   Distance_to_Store  1000 non-null   int64  
 7   Vegetarian         1000 non-null   int64  
 8   N_Children         1000 non-null   int64  
 9   Family_Pet         1000 non-null   int64  
dtypes: float64(2), int64(8)
memory usage: 78.3 KB


In [14]:
# Dropping null rows

drop_data = data.dropna().reset_index()
drop_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 977 entries, 0 to 976
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   index              977 non-null    int64  
 1   Grocery_Bill       977 non-null    float64
 2   N_Adults           977 non-null    int64  
 3   Family_Income      977 non-null    float64
 4   Family_Size        977 non-null    int64  
 5   N_Vehicles         977 non-null    int64  
 6   Distance_to_Store  977 non-null    int64  
 7   Vegetarian         977 non-null    int64  
 8   N_Children         977 non-null    int64  
 9   Family_Pet         977 non-null    int64  
dtypes: float64(2), int64(8)
memory usage: 76.5 KB


In [15]:
model_drop = ols(frmla,drop_data).fit()

In [16]:
print(model_drop.summary())

                            OLS Regression Results                            
Dep. Variable:           Grocery_Bill   R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     723.7
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        19:39:19   Log-Likelihood:                -4788.7
No. Observations:                 977   AIC:                             9593.
Df Residuals:                     969   BIC:                             9633.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            23.0180      5.79

In [ ]:
# Imputed results arent much different from the dropped results.

In [18]:
#Import the required libraries
import statsmodels.imputation.mice as mice
import statsmodels.regression.linear_model as sm

#Impute the data set with MICE Data
imp = mice.MICEData(data)
imputed_data = imp.next_sample()

#Merge the imputed data with the blank data
merge_data = pd.merge(imputed_data,data,left_index=True,right_index=True)
merge_data = merge_data[['Family_Income_x','Family_Income_y']]
#Family_Income_x are the imputed values
merge_data[(pd.isnull(merge_data.Family_Income_y))]

,Family_Income_x,Family_Income_y
11,86040.0,NaN
27,91569.0,NaN
55,75195.0,NaN
72,111183.0,NaN
93,120257.0,NaN
123,141806.0,NaN
143,83805.0,NaN
161,89335.0,NaN
182,75545.0,NaN
201,80429.0,NaN


In [19]:
mice = mice.MICE(frmla, sm.OLS, imp)
results = mice.fit(n_imputations=10)
print(results.summary())

                              Results: MICE
Method:                   MICE              Sample size:          1000   
Model:                    OLS               Scale                 1066.97
Dependent variable:       Grocery_Bill      Num. imputations      10     
-------------------------------------------------------------------------
                   Coef.  Std.Err.    t    P>|t|   [0.025   0.975]  FMI  
-------------------------------------------------------------------------
Intercept         23.5172   5.7491  4.0906 0.0000  12.2491 34.7853 0.0053
N_Adults          54.7917   3.8463 14.2453 0.0000  47.2530 62.3303 0.0119
Family_Income      0.0009   0.0001 10.5720 0.0000   0.0008  0.0011 0.0151
N_Vehicles        -0.7755   1.1518 -0.6733 0.5007  -3.0329  1.4819 0.0028
Distance_to_Store  3.8899   0.1911 20.3548 0.0000   3.5153  4.2645 0.0015
Vegetarian        -7.8137   4.2585 -1.8348 0.0665 -16.1602  0.5329 0.0001
N_Children        28.3621   1.2192 23.2623 0.0000  25.9724 30.7517 0